# NER Baseline
This pipeline uses BERT-like transformers on the NER dataset to collect performance metrics on finetuning

## Pipeline

In [ ]:
from collections import Counter
import pandas as pd

from util import convert_dataset_to_hf

import matplotlib.pyplot as plt

df = pd.read_json("../data/labelled/all.jsonl", lines=True)
clean_samples = convert_dataset_to_hf(df)
tmp = [l["ner_tags"] for l in clean_samples]
# flatten tmp
tmp = [item for sublist in tmp for item in sublist]
# get unique labels
labels = Counter(tmp)


# Plot labels Counter excluding O tag
labels = Counter({k: v for k, v in labels.items() if k != "O"})

# Plot
plt.figure(figsize=(10, 5))
plt.bar(labels.keys(), labels.values(), color="skyblue")
plt.xlabel("NER Label")
plt.ylabel("Count")
plt.title("Distribution of NER Labels (excluding O tags)")
plt.tight_layout()
plt.show()

In [ ]:
label2id = {
    "O": 0,
    "B-ORG": 1,
    "I-ORG": 2,
    "B-PER": 3,
    "I-PER": 4,
    "B-LOC": 5,
    "I-LOC": 6,
    "B-MISC": 7,
    "I-MISC": 8,
}
id2label = {v: k for k, v in label2id.items()}

In [ ]:
from datasets import load_dataset, Features, Value, ClassLabel, Sequence

features = Features(
    {
        "id": Value(dtype="string", id=None),
        "tokens": Sequence(feature=Value(dtype="string", id=None), id=None),
        "ner_tags": Sequence(ClassLabel(names=list(label2id.keys()))),
    }
)

ds = load_dataset(
    "json",
    data_files="../data/labelled/all_hf.jsonl",
    features=features,
    split="train",
)

dataset = ds.train_test_split(test_size=0.3)
dataset

In [ ]:
label_list = dataset["train"].features["ner_tags"].feature.names

label_list

In [ ]:
from transformers import AutoTokenizer

model_name = "distilbert/distilbert-base-uncased"
# model_name = "bullmount/hseBert-it-cased"
# model_name = "FacebookAI/roberta-base"
# model_name = "dbmdz/bert-base-italian-xxl-cased"  # Usato in it_nerIta_trf

tokenizer = AutoTokenizer.from_pretrained(model_name, add_prefix_space=True)

In [ ]:
def tokenize_and_align_labels(examples):
    tokenized_inputs = tokenizer(
        examples["tokens"],
        truncation=True,
        is_split_into_words=True,
    )

    labels = []
    for i, label in enumerate(examples["ner_tags"]):
        word_ids = tokenized_inputs.word_ids(
            batch_index=i
        )  # Map tokens to their respective word.
        previous_word_idx = None
        label_ids = []
        for word_idx in word_ids:  # Set the special tokens to -100.
            if word_idx is None:
                label_ids.append(-100)
            elif (
                word_idx != previous_word_idx
            ):  # Only label the first token of a given word.
                label_ids.append(label[word_idx])
            else:
                label_ids.append(-100)
            previous_word_idx = word_idx
        labels.append(label_ids)

    tokenized_inputs["labels"] = labels
    return tokenized_inputs

In [ ]:
tokenized_ds = dataset.map(tokenize_and_align_labels, batched=True)

In [ ]:
from transformers import DataCollatorForTokenClassification

data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)

In [ ]:
import evaluate

seqeval = evaluate.load("seqeval")

In [ ]:
import numpy as np


def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)

    true_predictions = [
        [label_list[p] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]
    true_labels = [
        [label_list[l] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]

    results = seqeval.compute(predictions=true_predictions, references=true_labels)
    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
        "accuracy": results["overall_accuracy"],
    }

In [ ]:
from transformers import AutoModelForTokenClassification, Trainer, TrainingArguments

model = AutoModelForTokenClassification.from_pretrained(
    model_name,
    num_labels=len(label2id),
    label2id=label2id,
    id2label=id2label,
)

In [ ]:
print(f"Starting training on {model_name}")
training_args = TrainingArguments(
    output_dir=f"../models/{model_name.split('/')[-1]}",
    num_train_epochs=10,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    logging_steps=10,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_ds["train"],
    eval_dataset=tokenized_ds["test"],
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

trainer.train()

In [ ]:
import os
import shutil

trainer.save_model(f"../models/{model_name.split('/')[-1]}/best_model")

# Delete non-best checkpoints
for file in os.listdir(f"../models/{model_name.split('/')[-1]}"):
    if file.startswith("checkpoint"):
        shutil.rmtree(f"../models/{model_name.split('/')[-1]}/{file}")

In [ ]:
import json

metrics = trainer.evaluate()
with open(f"../models/{model_name.split('/')[-1]}/best_model/metrics.json", "w") as f:
    json.dump(metrics, f, indent=4)

In [ ]:
with open("../output/experiments_results.json") as file:
    res = json.load(file)

res[model_name.split("/")[-1]] = metrics

# Append results to json
with open("../output/experiments_results.json", "w") as file:
    json.dump(res, file, indent=4)

In [ ]:
import json
import matplotlib.pyplot as plt

with open("../output/experiments_results.json") as file:
    res = json.load(file)

# Fixed order of metrics
metrics = ["precision", "recall", "f1"]
n_metrics = len(metrics)
models = list(res.keys())
x = np.arange(len(models))  # One x-position per model
width = 0.18  # Width of individual bars

fig, ax = plt.subplots(figsize=(10, 6))


# For each metric, plot across all models
for i, metric in enumerate(metrics):
    metric_values = []
    for model in models:
        model_data = res[model]
        if metric in model_data:
            metric_values.append(model_data[metric])
        elif f"eval_{metric}" in model_data:
            metric_values.append(model_data[f"eval_{metric}"])
        else:
            metric_values.append(0.0)  # or np.nan if you prefer
    bar_positions = x + (i - n_metrics / 2 + 0.5) * width
    ax.bar(bar_positions, metric_values, width, label=metric.capitalize())

# Labeling
ax.set_ylabel("Score")
ax.set_title("Metrics per Model")
ax.set_xticks(x)

# Set text size on x axis
for tick in ax.get_xticklabels():
    tick.set_fontsize(8)

# Show 1.0 on y axis

ax.set_xticklabels(models)
ax.legend(title="Metric")

plt.tight_layout()
plt.show()

## Inference

In [ ]:
from transformers import pipeline

pl = pipeline(
    "ner",
    model=f"../models/{model_name.split('/')[-1]}/best_model",
    aggregation_strategy="first",
)

In [ ]:
text = " ".join(dataset["test"][0]["tokens"])
print(text)
pl((text))